In [3]:
#persiapan dataset
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


In [4]:
#SparkSession dan baca CSV
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count

spark = SparkSession.builder \
    .appName("Tugas12-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# inferSchema=True wajib: tanpa ini qty & waktu_tunggu_menit terbaca string
# dan VectorAssembler akan error
df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 11:47:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 5 rows



In [5]:
# A. Membangun ML Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline

idx_menu   = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu  = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

ohe = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

assembler = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])
print("Pipeline Logistic Regression didefinisikan (6 tahap).")

Pipeline Logistic Regression didefinisikan (6 tahap).


In [6]:
# B. Pelatihan & Evaluasi
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

train, test = df.randomSplit([0.8, 0.2], seed=77)
print(f"Train: {train.count()} | Test: {test.count()}")

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1  = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(pred_lr)
f1_lr  = ev_f1.evaluate(pred_lr)
auc_lr = ev_auc.evaluate(pred_lr)

print("=== Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix:")
pred_lr.groupBy("ulasan_positif", "prediction").count() \
       .orderBy("ulasan_positif", "prediction").show()

Train: 2411 | Test: 589


=== Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix:


[Stage 49:>                                                         (0 + 1) / 1]

+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



In [7]:
# C. Model Pembanding
# labelCol WAJIB diset ke "ulasan_positif" (default-nya "label" dan akan error)
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)

# Lima tahap awal identik dengan bagian A, hanya tahap terakhir diganti
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(pred_rf)
f1_rf  = ev_f1.evaluate(pred_rf)
auc_rf = ev_auc.evaluate(pred_rf)

print("=== Perbandingan Hasil (data test) ===")
print(f"{'Model':<22}{'Accuracy':>10}{'F1':>10}{'AUC':>10}")
print(f"{'Logistic Regression':<22}{acc_lr:>10.4f}{f1_lr:>10.4f}{auc_lr:>10.4f}")
print(f"{'Random Forest':<22}{acc_rf:>10.4f}{f1_rf:>10.4f}{auc_rf:>10.4f}")

=== Perbandingan Hasil (data test) ===
Model                   Accuracy        F1       AUC
Logistic Regression       0.7929    0.7926    0.8650
Random Forest             0.7946    0.7942    0.8413


In [8]:
# D. Analisis Kesalahan Model
# Pilih model terbaik berdasarkan AUC tertinggi
if auc_rf > auc_lr:
    nama_terbaik, pred_terbaik = "Random Forest", pred_rf
else:
    nama_terbaik, pred_terbaik = "Logistic Regression", pred_lr
print(f"Model terbaik (AUC tertinggi): {nama_terbaik}")

# 10 baris yang salah diprediksi
salah = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))
print(f"Total salah prediksi: {salah.count()} dari {pred_terbaik.count()}")
salah.select("nama_menu", "cabang", "waktu_pesan", "qty",
             "waktu_tunggu_menit", "ulasan_positif", "prediction").show(10)

# Pola: tingkat kesalahan per nilai waktu tunggu
pred_terbaik.groupBy("waktu_tunggu_menit").agg(
    count("*").alias("jumlah"),
    avg(when(col("ulasan_positif") != col("prediction"), 1).otherwise(0)).alias("error_rate")
).orderBy("waktu_tunggu_menit").show(25)

Model terbaik (AUC tertinggi): Logistic Regression
Total salah prediksi: 122 dari 589
+---------+--------+-----------+---+------------------+--------------+----------+
|nama_menu|  cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|
+---------+--------+-----------+---+------------------+--------------+----------+
|Americano|Magelang|      Malam|  1|                15|             1|       0.0|
|Americano|Magelang|      Siang|  2|                22|             1|       0.0|
|Americano|Magelang|       Sore|  1|                 7|             0|       1.0|
|Americano|Magelang|       Sore|  2|                11|             0|       1.0|
|Americano|Magelang|       Sore|  3|                 5|             0|       1.0|
|Americano|Magelang|       Sore|  3|                13|             1|       0.0|
|Americano|Magelang|       Sore|  5|                 6|             0|       1.0|
|Americano|Magelang|       Sore|  5|                15|             1|       0.0|
|Americano|S

In [9]:
# E. Kesimpulan Bisnis
# Nama fitur sesuai urutan VectorAssembler (OHE membuang kategori terakhir / dropLast)
nama_fitur = []
for stage_idx, prefix in [(0, "menu"), (1, "cabang"), (2, "waktu")]:
    labels = model_lr.stages[stage_idx].labelsArray[0]
    nama_fitur += [f"{prefix}={l}" for l in labels[:-1]]
nama_fitur += ["qty", "waktu_tunggu_menit"]

koef = model_lr.stages[-1].coefficients.toArray()
for nama, k in sorted(zip(nama_fitur, koef), key=lambda x: -abs(x[1])):
    print(f"{nama:<28}{k:>9.4f}")

waktu=Malam                    0.2530
waktu_tunggu_menit            -0.2220
waktu=Sore                     0.1857
menu=Roti Bakar               -0.1443
menu=Kopi Susu                 0.1171
cabang=Semarang               -0.1140
menu=Americano                -0.0855
waktu=Pagi                    -0.0651
cabang=Yogyakarta              0.0645
menu=Cappuccino                0.0194
menu=Es Teh                   -0.0191
qty                           -0.0108


# E. Kesimpulan Bisnis
Model yang dibangun dapat menunjukkan bahwa waktu tunggu pesanan adalah faktor yang paling menunjukkan apakah pelanggan memberikan ulasan positif. Sementara itu Logistic Regression, memprediksi ulasan positif dengan benar pada sekitar 79% data uji (467 dari 589 pesanan). Fitur yang paling berpengaruh adalah waktu tunggu pesanan: semakin lama pelanggan menunggu, semakin kecil peluang ulasan positif. Fitur lain seperti menu, cabang, waktu pesan, dan qty pengaruhnya jauh lebih kecil.

Analisis kesalahan pada bagian D menunjukkan model paling sering keliru pada waktu tunggu sekitar 10–13 menit, dengan error rate 35–58%, sedangkan pada waktu tunggu sangat singkat (2–5 menit) error rate-nya rendah. Artinya, rentang 10–13 menit adalah zona abu-abu: pelanggan bisa puas atau kecewa dan perilakunya paling sulit diprediksi.

Rekomendasi untuk manajemen Kopi Nusantara:
1. Tetapkan target waktu penyajian maksimal ±10 menit di semua cabang, karena setelah titik ini peluang ulasan positif turun tajam.
2. Beri notifikasi kepada barista/kasir saat pesanan melewati 10 menit agar segera diprioritaskan.
3. Siapkan kompensasi kecil (mis. diskon atau minuman gratis) untuk pesanan yang menunggu lebih dari 15 menit.
4. Fokuskan evaluasi proses dapur pada pesanan yang berada di zona 10–13 menit, karena perbaikan kecil di sini paling mungkin mengubah kesan pelanggan.

In [16]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

for v in ["train", "test", "idx_menu", "idx_cabang", "idx_waktu", "ohe", "assembler",
          "ev_acc", "ev_f1", "ev_auc", "model_lr", "model_rf",
          "pred_lr", "pred_rf", "auc_lr", "auc_rf", "pipeline_rf", "rf"]:
    assert v in globals(), f"Variabel '{v}' belum ada, jalankan dulu bagian A-C"
print("Siap, digunakan")

Siap, digunakan


In [17]:
# Tambahan 
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

param_grid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [10, 30, 50])
    .addGrid(rf.maxDepth, [3, 5, 8])
    .build())

cv = CrossValidator(
    estimator=pipeline_rf,          # seluruh Pipeline ikut di-cross-validate
    estimatorParamMaps=param_grid,
    evaluator=ev_auc,
    numFolds=3,
    seed=77
)

cv_model = cv.fit(train)            # 9 kombinasi x 3 fold = 27 kali pelatihan, bisa agak lama

best_rf = cv_model.bestModel.stages[-1]
print("numTrees terbaik :", best_rf.getNumTrees)
print("maxDepth terbaik :", best_rf.getOrDefault("maxDepth"))
print("AUC rata-rata CV :", max(cv_model.avgMetrics))

pred_cv = cv_model.transform(test)
print("AUC pada data test:", ev_auc.evaluate(pred_cv))

numTrees terbaik : 50
maxDepth terbaik : 5
AUC rata-rata CV : 0.8147918066710638
AUC pada data test: 0.8463346844238572


In [18]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="features_scaled",
                        withStd=True, withMean=False)   # withMean=False karena vektor sparse
lr_scaled = LogisticRegression(featuresCol="features_scaled", labelCol="ulasan_positif")

pipe_scaled = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, scaler, lr_scaled])
pred_scaled = pipe_scaled.fit(train).transform(test)

print(f"LR tanpa scaler  -> Accuracy: {ev_acc.evaluate(pred_lr):.4f} | AUC: {auc_lr:.4f}")
print(f"LR dengan scaler -> Accuracy: {ev_acc.evaluate(pred_scaled):.4f} | AUC: {ev_auc.evaluate(pred_scaled):.4f}")

# Regularisasi: L2 (Ridge), Elastic Net, L1 (Lasso)
for nama, kw in [("L2 (Ridge)", dict(regParam=0.1, elasticNetParam=0.0)),
                 ("Elastic Net", dict(regParam=0.1, elasticNetParam=0.5)),
                 ("L1 (Lasso)", dict(regParam=0.1, elasticNetParam=1.0))]:
    m = LogisticRegression(featuresCol="features", labelCol="ulasan_positif", **kw)
    p = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, m]).fit(train).transform(test)
    print(f"{nama:<12} -> AUC: {ev_auc.evaluate(p):.4f}")

LR tanpa scaler  -> Accuracy: 0.7929 | AUC: 0.8650
LR dengan scaler -> Accuracy: 0.7929 | AUC: 0.8650
L2 (Ridge)   -> AUC: 0.8654
Elastic Net  -> AUC: 0.8662
L1 (Lasso)   -> AUC: 0.8662


In [19]:
# Nama fitur sesuai urutan VectorAssembler (OHE membuang kategori terakhir / dropLast=True)
nama_fitur = []
for i, prefix in enumerate(["menu", "cabang", "waktu"]):
    labels = model_lr.stages[i].labelsArray[0]
    nama_fitur += [f"{prefix}={l}" for l in labels[:-1]]
nama_fitur += ["qty", "waktu_tunggu_menit"]

koef = model_lr.stages[-1].coefficients.toArray()
imp  = model_rf.stages[-1].featureImportances.toArray()
assert len(nama_fitur) == len(koef) == len(imp), "Jumlah nama fitur tidak sama dengan jumlah koefisien"

tabel = pd.DataFrame({"fitur": nama_fitur, "koef_LR": koef, "importance_RF": imp})
print(tabel.sort_values("importance_RF", ascending=False).round(4).to_string(index=False))
print("\nTotal importance RF:", round(imp.sum(), 4))

             fitur  koef_LR  importance_RF
waktu_tunggu_menit  -0.2220         0.9460
               qty  -0.0108         0.0118
       waktu=Malam   0.2530         0.0075
    menu=Kopi Susu   0.1171         0.0072
 cabang=Yogyakarta   0.0645         0.0060
   cabang=Semarang  -0.1140         0.0050
        waktu=Sore   0.1857         0.0040
   menu=Roti Bakar  -0.1443         0.0039
    menu=Americano  -0.0855         0.0025
       menu=Es Teh  -0.0191         0.0023
        waktu=Pagi  -0.0651         0.0023
   menu=Cappuccino   0.0194         0.0016

Total importance RF: 1.0


In [20]:
from pyspark.ml.functions import vector_to_array

# Precision/recall/F1 kelas 1 lewat evaluator
for metrik in ["precisionByLabel", "recallByLabel", "fMeasureByLabel"]:
    ev = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction",
                                           metricName=metrik, metricLabel=1.0)
    print(f"{metrik:<18} (kelas 1): {ev.evaluate(pred_lr):.4f}")

ev_pr = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderPR")
print("AUC-PR:", round(ev_pr.evaluate(pred_lr), 4))

# Pengaruh threshold: dihitung dari probabilitas kelas 1 (tanpa mengubah model)
skor = pred_lr.select(F.col("ulasan_positif").alias("y"),
                      vector_to_array("probability")[1].alias("p1"))

print("\nthreshold | precision | recall | TP  FP  FN")
for t in [0.3, 0.5, 0.7]:
    r = skor.select(
        F.sum(F.when((F.col("p1") > t) & (F.col("y") == 1), 1).otherwise(0)).alias("tp"),
        F.sum(F.when((F.col("p1") > t) & (F.col("y") == 0), 1).otherwise(0)).alias("fp"),
        F.sum(F.when((F.col("p1") <= t) & (F.col("y") == 1), 1).otherwise(0)).alias("fn"),
    ).first()
    tp, fp, fn = int(r["tp"]), int(r["fp"]), int(r["fn"])
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    print(f"   {t:.1f}    |  {prec:.4f}   | {rec:.4f} | {tp:>3} {fp:>3} {fn:>3}")

precisionByLabel   (kelas 1): 0.7865
recallByLabel      (kelas 1): 0.7636
fMeasureByLabel    (kelas 1): 0.7749
AUC-PR: 0.8463

threshold | precision | recall | TP  FP  FN
   0.3    |  0.6936   | 0.9055 | 249 110  26


   0.5    |  0.7865   | 0.7636 | 210  57  65
   0.7    |  0.8508   | 0.5600 | 154  27 121


In [21]:
import numpy as np
from pyspark.ml.functions import vector_to_array

dt = pred_lr.select(F.col("ulasan_positif").alias("y"),
                    vector_to_array("probability")[1].alias("p1")).toPandas()
y = dt["y"].to_numpy().astype(int)
s = dt["p1"].to_numpy()

urut = np.argsort(-s, kind="mergesort")
ys, ss = y[urut], s[urut]
tp = np.cumsum(ys)
fp = np.cumsum(1 - ys)
akhir = np.r_[np.where(np.diff(ss) != 0)[0], len(ss) - 1]     # satu titik per skor unik
tpr = np.r_[0.0, tp[akhir] / ys.sum()]
fpr = np.r_[0.0, fp[akhir] / (len(ys) - ys.sum())]
auc_manual = float(np.sum((fpr[1:] - fpr[:-1]) * (tpr[1:] + tpr[:-1]) / 2))
print(f"AUC manual = {auc_manual:.4f} | AUC evaluator = {auc_lr:.4f}")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 5))
    plt.plot(fpr, tpr, label=f"Logistic Regression (AUC={auc_manual:.3f})")
    plt.plot([0, 1], [0, 1], "k--", label="Tebakan acak")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Kurva ROC - Data Test")
    plt.legend()
    plt.show()
except ImportError:
    print("matplotlib belum terpasang. Jalankan: pip install matplotlib")

AUC manual = 0.8650 | AUC evaluator = 0.8650


<Figure size 500x500 with 1 Axes>

In [22]:
model_lr.write().overwrite().save("model_lr_tugas12")
model_dimuat = PipelineModel.load("model_lr_tugas12")

auc_dimuat = ev_auc.evaluate(model_dimuat.transform(test))
print(f"AUC model asli   : {auc_lr:.6f}")
print(f"AUC model dimuat : {auc_dimuat:.6f}")
print("Sama persis" if abs(auc_lr - auc_dimuat) < 1e-9 else "Berbeda, periksa kembali")

AUC model asli   : 0.865009
AUC model dimuat : 0.865009
Sama persis


In [23]:
baru = spark.createDataFrame(
    [("Matcha Latte", "Magelang", "Pagi", 2, 8),
     ("Kopi Susu", "Semarang", "Sore", 1, 15)],
    ["nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit"])

# Pipeline awal memakai handleInvalid default ("error")
try:
    model_lr.transform(baru).select("nama_menu", "prediction").show()
except Exception as e:
    print("Default (error) gagal pada menu baru:", type(e).__name__)

# Versi aman: handleInvalid="keep" pada StringIndexer dan OneHotEncoder
idx_menu_k   = StringIndexer(inputCol="nama_menu", outputCol="menu_idx", handleInvalid="keep")
idx_cabang_k = StringIndexer(inputCol="cabang", outputCol="cabang_idx", handleInvalid="keep")
idx_waktu_k  = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx", handleInvalid="keep")
ohe_k = OneHotEncoder(inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
                      outputCols=["menu_vec", "cabang_vec", "waktu_vec"], handleInvalid="keep")
lr_k = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")
model_aman = Pipeline(stages=[idx_menu_k, idx_cabang_k, idx_waktu_k, ohe_k, assembler, lr_k]).fit(train)
model_aman.transform(baru).select("nama_menu", "waktu_tunggu_menit", "prediction").show()

26/11/19 11:58:39 ERROR Executor: Exception in task 0.0 in stage 4329.0 (TID 3365)
org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] Failed to execute user defined function (`StringIndexerModel$$Lambda$3754/0x000000010168e040`: (string) => double).
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:198)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:43)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$getByteArrayRdd$1(SparkPlan.scala:388)
	at org.apache.spark.rdd.RDD.$a

Default (error) gagal pada menu baru: Py4JJavaError


+------------+------------------+----------+
|   nama_menu|waktu_tunggu_menit|prediction|
+------------+------------------+----------+
|Matcha Latte|                 8|       1.0|
|   Kopi Susu|                15|       0.0|
+------------+------------------+----------+



In [24]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
